In [1]:
import requests
from bs4 import BeautifulSoup
import re

In [2]:
division = "1"
sport_code = "MBB"
academic_year = "2025"

NCAA_BASE = "https://stats.ncaa.org"
NCAA_HEADERS = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10.15; rv:147.0) Gecko/20100101 Firefox/147.0",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
    "Referer": f"{NCAA_BASE}/",
    "Connection": "keep-alive",
}

params = {
    "division": division,
    "sport_code": sport_code,
    "academic_year": academic_year,
}

# Use a Session (like load.ncaa.core) - NCAA site often requires session cookies
session = requests.Session()
session.headers.update(NCAA_HEADERS)
session.get(NCAA_BASE)  # establish session / cookies
response = session.get(NCAA_BASE + "/team/inst_team_list", params=params)

In [3]:
soup = BeautifulSoup(response.content)

In [12]:
"/teams/" in soup.find_all("a", href=True)[-50]["href"] 

True

In [15]:
soup.find_all("a", href=True)[-50]

<a href="/teams/590556">UMBC</a>

In [5]:
# Parse team list like team_list.py does
import pandas as pd
from load.ncaa.core import table_to_df, extract_links

df = table_to_df(soup)
if df.empty:
    links = extract_links(soup, r"org_id=\d+|/team/\d+")
    df = pd.DataFrame(links, columns=["team_name", "team_href"])
df

ImportError: cannot import name 'client' from 'load.ncaa.client' (/Users/spencerklug/projects/nba-stats/load/ncaa/client.py)

In [ ]:
# Parse team list like team_list.py does (extract links matching org_id or /team/)
import re
team_links = [(a.get_text(strip=True), a.get("href", "")) for a in soup.find_all("a", href=True) if re.search(r"org_id=\d+|/team/\d+", a.get("href", ""))]
print(f"Found {len(team_links)} teams")
team_links[:10]  # first 10

In [ ]:
# Parse team list like team_list.py does (table or org_id/team links)
import pandas as pd

def table_to_df(s):
    table = s.select_one("table")
    if not table:
        return pd.DataFrame()
    rows = table.find_all("tr")
    if not rows:
        return pd.DataFrame()
    headers = [c.get_text(strip=True) for c in rows[0].find_all(["th", "td"])]
    data = [[c.get_text(strip=True) for c in tr.find_all(["td", "th"])] for tr in rows[1:]]
    return pd.DataFrame(data, columns=headers) if data else pd.DataFrame(columns=headers)

def extract_links(s, pattern):
    out = []
    for a in s.find_all("a", href=True):
        if re.search(pattern, a.get("href", "")):
            out.append((a.get_text(strip=True), a.get("href", "")))
    return out

df = table_to_df(soup)
if df.empty:
    links = extract_links(soup, r"org_id=\d+|/team/\d+")
    df = pd.DataFrame(links, columns=["team_name", "team_href"])
df